In [ ]:
import os
import sys
import json
import pandas as pd

from irat.utils.settings import env



# —————— Configuration ——————
LLM_name = env("LLM_NAME")
print(f"LLM: {LLM_name}")

# ensure project root is on PYTHONPATH
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

def switch_to_dir(eval_name: str):
    # climb up until you find README.md, then cd into evaluation/<eval_name>-responses-<LLM>
    while not os.path.exists("README.md"):
        os.chdir("..")
    os.chdir("evaluation")
    folder = f"{eval_name}-responses-{LLM_name.replace(':','-').replace('/','-')}"
    print(f"-> entering {folder}")
    os.chdir(folder)

# —————— Main loop ——————
records = []

for ds in ["human_eval","mbpp","gsm8k"]:
    switch_to_dir(ds)
    for fn in os.listdir():
        if not fn.endswith(".json"):
            continue
        with open(fn, "r") as f:
            data = json.load(f)
        records.append({
            "dataset":      ds,
            "task_id":      data.get("task_id"),
            "old_rat_retrievals": data['responses'][0].get("old_rat_retrievals", 0),
            "retrievals":       data['responses'][0].get("retrievals", 0),
        })
    # go back up to evaluation/ so the next switch_to_dir works
    os.chdir(os.path.join(".."))

# —————— Build DataFrame & Display ——————
df = pd.DataFrame(records)

# 2) Aggregate averages
print("\nAverage retrievals by dataset:\n")
avg_retr_df = (
	df.groupby("dataset")[["old_rat_retrievals","retrievals"]]
        .mean()
        .round(2)
        .rename(columns={
            "old_rat_retrievals":"avg_old_rat_retrievals",
            "retrievals":"avg_retrievals"
        })
)
avg_retr_df["avg_old_rat_retrievals"] = avg_retr_df["avg_old_rat_retrievals"].astype(float).round(2)
avg_retr_df["avg_retrievals"] = avg_retr_df["avg_retrievals"].astype(float).round(2)
# compare "saved" like 30% the RAT retrievals
avg_retr_df["saved"] = ((avg_retr_df["avg_old_rat_retrievals"] - avg_retr_df["avg_retrievals"]) / avg_retr_df["avg_old_rat_retrievals"] * 100).round(2).astype(str) + '%'
print(avg_retr_df.reset_index().to_string())


LLM: Llama-3.3-70B
-> entering human_eval-responses-Llama-3.3-70B
-> entering mbpp-responses-Llama-3.3-70B
-> entering gsm8k-responses-Llama-3.3-70B

Average retrievals by dataset:

      dataset  avg_old_rat_retrievals  avg_retrievals   saved
0       gsm8k                    3.43            1.76  48.69%
1  human_eval                    4.46            3.16  29.15%
2        mbpp                    5.24            3.36  35.88%
